# Integrated Intelligent Systems: Tutorial 3 - Body Motion Problem (Scientific Object) (3 Pts)

This notebook introduces a formulation body motion problem, central to designing and implementing Cognition-Enabled Robot Agents, as a description logic system.

## Part 1: Robot-World Manipulation System as Scientific Object (E1–E6) (1.5 Pt)

**Title:** Formalizing the Robot Agent: From Architecture to Description Logic Reasoners

![Alternative Text](RWMS.png)

### Scenario Background
An autonomous robotic agent (the PR2 in the EASE apartment) is deployed to execute a manipulation task ("Serve Eva a drink"). To validate this agent scientifically, we must formalize the system as a **Robot-World Manipulation System (RWMS)**, separate the controlling subsystem from the controlled body-world system, define trajectories as trees, and assess runs against an external problem standard using a description logic reasoner.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)

1. **E1 · Architectural Cut & Subsystem Separation**
   - In the graphical representation of the RWMS, a strict architectural "cut" is placed between the manipulation executive and the body-world system ($[B \bowtie W]$). Explain the nature of this cut: why is the manipulation executive classified as a `dul:Agent` (non-physical program) while the body-world system is classified under physical objects, and how do they interact exclusively across the boundary?

2. **E2 · Forward-Branching Time & World Choices**
   - During an operation on a fridge door or container seal, the same action token (e.g., a position-controlled pull) can lead to completely divergent physical outcomes depending on world choices (e.g., seal releases vs. seal holds tight). Draw or describe the structure of the resulting courses, and explain why the possible courses form a tree that branches *only forward* from the initial state $x_0$.

3. **E3 · Trajectory Classification & Priority Hierarchies**
   - The problem standard $P$ sorts every physical trajectory into one of four classes: **SOL** (Solution), **REPD** (Repaired), **NON** (Nonachievement), and **PROH** (Prohibited). Consider two runs that both successfully place the drink within Eva's reach, but where Run A tears off the container handle (`P4` violation) while Run B maintains licensed contact. Classify both runs and explain why the classifier grades the *entire course* rather than just the final end-state.

4. **E4 · Description Logic Partonomy & DUL Constraints**
   - When formalizing the RWMS in description logic under DOLCE Ultra Lite (DUL), we define an RWMS as having *exactly one* executive and *exactly one* body-world system (`RWMS = (=1 hasComponent.ManipExecutive) ⊓ (=1 hasComponent.BodyWorldSystem)`). Using the transitivity of `dul:isPartOf`, explain how a reasoner (such as HermiT) automatically entails that the PR2's head camera is a component of the RWMS without explicitly hardcoding every part-whole link.

5. **E5 · Ontological Inconsistency & Entailment Limits**
   - An inconsistency in description logic serves as a strict analytical tool. Formulate a competency question that results in an ontological inconsistency when attempting to classify the *manipulation problem itself* as a physical component inside the RWMS. Furthermore, explain why numerical physical laws (such as Coulomb friction $2 \cdot \mu \cdot F \ge m \cdot (g + a)$) must be handled outside description logic by the semantic digital twin's causal model.

6. **E6 · Strength vs. Truth in Agent Validation**
   - The design thesis dictates that an agent decides within its believed causality, which can be tested for logical *strength* via competency queries and for *truth* via physical experiments. Compare how an axiom set that lacks a friction law ($\text{K}_0$) performs on competency queries versus an axiomatization containing explicit dynamical laws ($\text{K}_1/\text{K}_2$), and state what kind of test is required to validate the actual physical truth of those beliefs.

## Answers – Part 1 (E1–E6) | Lecture-aligned revision

### E1 · Architectural Cut & Subsystem Separation

Lecture 3 represents the **robot–world manipulation system (RWMS)** as **ME ↔ [B ⋈ W]**. The **manipulation executive (ME)** is the *controlling* subsystem: a computational agent (modelled as a non-physical `dul:Agent` at this abstraction level) which reads observations, interprets the task, selects alternatives and commits to **action tokens**. This does not imply that the program runs without physical computing hardware. The *controlled* subsystem is the coupled **body–world system**, with the PR2 body **B**, the apartment world **W**, and their contact and force interactions **⋈**. **Action tokens** pass from ME to BWS; **observations** return from BWS to ME. The executive cannot directly set the fridge state: the body must realize a token and the world decides the resulting physical transition. Importantly, the full RWMS should not be typed as `dul:PhysicalObject`: it includes a non-physical executive; the lecture classifies it more generally under `dul:Object`. The **manipulation problem P is an external evaluation standard**, not a component of the system.

*Source: Lecture 3 slides 10–11, 18, 31, 33–34; Lecture 4 slides 2, 5–7.*

### E2 · Forward-Branching Time & World Choices

The BWS is specified by $BWS_\theta = [B \bowtie W] = \langle S,A,O,D,T_\theta,\Omega_\theta,S_0\rangle$, with joint states $S=S_B\times S_W$. The executive selects an action; the world response depends on its state and disturbances (e.g. seal stiffness). A schematic forward-branching tree is:

```text
x0
├── executive: position-controlled pull
│   ├── world: seal releases → door opens
│   └── world: seal holds    → force rises → handle tears off (P4)
└── executive: force-limited pull
    ├── world: seal releases → door opens
    └── world: seal holds    → stop at F_max → decline/report (NON if goal missed)
```

The branches denote **alternative possible continuations**, not simultaneous outcomes. A history is shared until its first divergence; subsequent choices affect only the future. One actual execution follows one path. A **trajectory** includes the entire body–world course until the **task-relevant state has settled**, potentially after the last action token; a partial sensor record is evidence of the trajectory, not necessarily the full trajectory.

*Source: Lecture 3 slides 13–16, 23–24.*

### E3 · Trajectory Classification & Priority Hierarchies

The external problem standard $P$ maps every completed course to **one** category, in priority order **PROH → NON → REPD → SOL**. A run is accepted only in **SOL or REPD**.

- **Run A = PROH**: tearing off the handle violates the hard prevention **P4: no damage to protected components**, even though the drink ultimately reaches Eva. A forbidden event remains forbidden regardless of later success.
- **Run B = SOL**, *provided* all achievement conditions hold (e.g. the drink is stable and within Eva's reach), all maintenance conditions hold, and no prohibition is violated.
- **REPD** applies when the achievements are met with an *admitted, subsequently recovered* maintenance departure (e.g. a permitted temporary tilt that settles). **NON** applies if nothing prohibited occurs but an achievement is missed (e.g. safely declining to open a stuck fridge).

The classifier evaluates **achieve (final state)**, **maintain (over an interval)** and **prevent (throughout the whole course)**. It therefore cannot judge success from the final photograph alone.

*Source: Lecture 3 slides 18, 21–24, study slide 41.*

### E4 · Description Logic Partonomy & DUL Constraints

The exercise's exact-cardinality model is, in the lecture's notation:

$$RWMS \equiv (=1\;dul{:}hasComponent.ManipulationExecutive) \sqcap (=1\;dul{:}hasComponent.BodyWorldSystem).$$

Further, $BodyWorldSystem$ has one `RobotBody` component and at least one `WorldEntity`, while a `Sensor` is part of some `RobotBody`. For the **named head camera**, the ABox must link that camera to the PR2 body, and link the body to BWS and BWS to RWMS, using `dul:isPartOf` (or DUL part-property links). Since **`dul:isPartOf` is transitive**, HermiT can entail that the camera is part of the complete RWMS, without explicitly asserting the shortcut relation. The lecture notes the hierarchy **`dul:hasComponent ⊑ dul:hasProperPart ⊑ dul:hasPart`**. **Caution:** transitive `hasPart/isPartOf` reasoning is not identical to a direct `hasComponent` assertion; the reasoner does *not* infer that the head camera is a direct `hasComponent` of RWMS merely from the part chain. Moreover, OWL 2 does not permit exact cardinality on a transitive role: use `hasComponent` for the `=1` constraints and `hasPart`/`isPartOf` for chains.

*Source: Lecture 3 slides 31–34.*

### E5 · Ontological Inconsistency & Entailment Limits

**Competency question:** “Can a `ManipulationProblem` be entered as a **component** of an RWMS?” Add the test assertion `dul:hasComponent(rwms_1, problem_1)` where `ManipulationProblem(problem_1)` is an abstract problem/description. Lecture 3 gives the RWMS value restriction

$$RWMS \sqsubseteq \forall dul{:}hasComponent.(ManipulationExecutive \sqcup dul{:}PhysicalObject).$$

Because the ontology treats the manipulation problem as an abstract description disjoint from the permitted component types, HermiT finds the **augmented ontology inconsistent**. Merely asserting `assessedAgainst(rwms_1, problem_1)` is consistent and is in fact the intended relation (`RWMS ⊑ ∃assessedAgainst.ManipulationProblem`). This shows why the standard should remain **outside the RWMS** and why the conflict follows from the **specific value restriction and disjointness axioms** rather than from the intuitive word “abstract” alone.

A DL reasoner answers class membership, existential, cardinality and partonomy questions; it cannot calculate real-valued friction/contact dynamics. For a simplified two-sided pinch the grip condition is $2\mu F_n \ge m(g+a)$. The semantic digital twin's **causal/physical model** must evaluate the arithmetic with specified parameters and assumptions. Lecture 3's illustrative case of a wet 0.64 kg carton, $\mu=0.19$ and an 18 N pinch is a **teaching assumption**, not a measured PR2 result.

*Source: Lecture 3 slides 18, 31, 33–34; Lecture 4 slide 2.*

### E6 · Strength vs. Truth in Agent Validation

The course distinguishes **logical strength** from **empirical truth**. A **K0** description without the friction relation can still answer ontological competency queries (e.g. components, observed data types, existence of a relevant sensor), but it has **insufficient information** to determine whether a wet carton slips in a specific lift. A richer **K1/K2** account that links the ontology to an appropriately parameterized physical causal law can yield quantitative predictions and support stronger competency questions, e.g. whether a proposed normal force exceeds the predicted slip threshold. The labels K0/K1/K2 here follow the exercise wording; the important distinction is what laws and parameters each knowledge account actually contains—not the names alone.

Such a stronger theory is not necessarily **true**. Validation requires predictions made *before* tests, physical/simulation experiments under declared conditions, measured grip forces, masses, friction regimes, accelerations and slips, and a comparison of observations to predictions across varied cases. A successful demonstration is weaker evidence than a supported competence claim on a declared scope. **Competency questions test strength; experiments test truth.**

*Source: Lecture 3 slides 35–38; Lecture 3 student chapter sections on scientific/engineering/evidential commitments; Lecture 4 student chapter, physical-body/world modelling.*

## Part 2: Robot Motion Problem and Description Logics (1.5 Pt)

**Title:** Formulating and Querying the Robot Motion Problem with Knowledge Bases & Description Logics

![Alternative Text](PR2.png)

### Scenario Background
Using the interactive Jupyter notebook and RViz simulation environment (`bmp.rviz`, `setup_demo()`), the PR2 mobile manipulator interacts with the EASE apartment model (including the fridge and various kitchen drawers). Instead of hardcoding trajectories, the robotic executive queries a knowledge base (represented via Description Logics / Prolog reasoning) to link task requests to state changes, causes (motions), and capabilities.

[Go to the programming Lab](https://binder.intel4coro.de/v2/gh/maltehue/giskard-examples/bmp_vrb?urlpath=/lab/tree/notebooks/bodyMotionProblem.ipynb)
---

### Multi-Question Student Task Sheet (0.25 Pt x 6)

1. **E7 · Knowledge Base Grounding & Task Requests**
   - Execute the initial query `taskRequest(Request, StateChange).`Examine the resulting solution bindings (e.g., `{'Request': 'open_fridge', 'StateChange': 'open(fridge)'}` and various drawer operations). Explain how mapping abstract task requests to discrete state changes forms the top-level specification layer for the manipulation executive.

2. **E8 · Extending Queries for Causal Motions & Parameters**
   - Extend the query to incorporate motion causes: `taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam).` Explain how the reasoner bridges the gap between a desired symbolic state change and continuous motion parameters (such as `envJointGoal`, target joint angles, and handle identifiers).

3. **E9 · Integrating Robot Capabilities (`canPerform`)**
   - Finalize the query by adding the robot capability constraint: `taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam), canPerform(Robot, Motion, MotionParam).`Describe what role the `Robot` variable (`'pr2'`) plays in filtering feasible trajectories through the reasoner.

4. **E10 · Specializing Request Scope (`open_fridge` vs. `open_anything`)**
   - Modify the query request parameter from general categories like `open_anything` to specifically target `open_fridge`. Explain how refining the scope in the knowledge base reduces the solution search tree and prevents unauthorized or unintended state changes in the kitchen environment.

5. **E11 · Visualization and Simulation Linkage in RViz**
   - With the background shell command executing `rviz -d .../bmp.rviz` and displaying the PR2 interacting with the fridge handle, explain how the motion parameters derived from the reasoner's output (`[0.8, b'fridge_door_joint', b'fridge_door_handle']`) are executed as physical joint trajectories in the simulation.

6. **E12 · Ontological Limits & Missing Dynamics**
   - While the knowledge base successfully infers *which* motion achieves a state change, explain why a pure Description Logic or symbolic Prolog reasoner cannot verify whether a specific grasp force will cause a wet carton to slip, and identify which subsystem must handle those physical calculations.

## Answers – Part 2 (E7–E12) | Lecture-aligned revision

**Verification boundary.** The exercise sheet supplies the predicate names and example bindings. The remote Binder knowledge base and RViz motion stack were **not executed** while preparing this submission. The Prolog queries are therefore executable *templates*, and any bindings shown below are **examples from the task sheet**, not claimed live results.

### E7 · Knowledge Base Grounding & Task Requests

```prolog
taskRequest(Request, StateChange).
```

This predicate maps the request **what to accomplish** to the intended symbolic **state change** (the problem/goal side), e.g. the task-sheet example `Request = open_fridge`, `StateChange = open(fridge)`. In lecture terminology, the request initially underdetermines the actual body motion. The mapping is a declarative **specification of intended effects**, *not* a robot trajectory, permission check or observed physical success. Grounding to the actual fridge and correct scene state still matters.

*Source: Exercise E7; Lecture 2 slides 5–12; Lecture 3 slides 10, 18–21.*

### E8 · Extending Queries for Causal Motions & Parameters

```prolog
taskRequest(Request, StateChange),
causes(Motion, StateChange, MotionParam).
```

A join on `StateChange` selects motions represented in the KB as causing the specified change. `Motion` is a symbolic motion class and `MotionParam` provides candidate parameters such as an `envJointGoal`, desired environment-joint value and the handle to manipulate. This is the **knowledge-to-action-designator step**: it proposes an actionable reference, but it does not directly solve inverse kinematics, validate the force limit, or establish that the seal releases. The actual transition depends on the coupled body–world system.

*Source: Exercise E8; Lecture 2 slides 8–12; Lecture 4 slides 5–7.*

### E9 · Integrating Robot Capabilities (`canPerform`)

```prolog
taskRequest(Request, StateChange),
causes(Motion, StateChange, MotionParam),
canPerform(Robot, Motion, MotionParam).
```

The final condition filters candidates against the **represented capabilities** of a robot. The variable `Robot` can bind to `pr2` for a KB fact that says PR2 can perform that motion. If the task is specifically for PR2, use `canPerform(pr2, Motion, MotionParam)` instead. This constitutes a **symbolic capability claim only**; actual configurations, collisions, joint limits, force/torque margins and controller behavior must still be checked. The lecture distinguishes what is **possible**, **achievable** with an executive and **actually achieved** on a recorded run.

*Source: Exercise E9; Lecture 3 slides 25–26, 42; Lecture 4 slides 8–16.*

### E10 · Specializing Request Scope (`open_fridge` vs. `open_anything`)

```prolog
taskRequest(open_fridge, StateChange),
causes(Motion, StateChange, MotionParam),
canPerform(pr2, Motion, MotionParam).
```

The fixed request restricts unification to the fridge-opening goal instead of admitting unrelated operations such as opening a drawer. This usually makes fewer candidate bindings available than a broad request like `open_anything` **if that general request is actually modelled in the KB**. Query scope alone is **not an authorization or safety mechanism**: the executive still needs explicit licensed-contact, prohibition, grounding and feasibility checks. In Lecture 3 the problem's **P4 damage prevention** applies regardless of which action is queried.

*Source: Exercise E10; Lecture 3 slides 21–24.*

### E11 · Visualization and Simulation Linkage in RViz

For the example parameters supplied by the exercise:

```python
[0.8, b'fridge_door_joint', b'fridge_door_handle']
```

`0.8` is the proposed target value of the fridge door's joint variable (for a revolute joint, normally in **radians**); `fridge_door_joint` identifies the environmental joint; `fridge_door_handle` identifies the intended contact/handle target. The intended execution chain is:

1. KB query selects a motion type and target parameters.
2. Motion planning/IK determines attainable hand poses and joint configurations using the PR2 kinematic description, including the handle frame.
3. A collision-aware controller coordinates the gripper/arm (and possibly base) with a trajectory compatible with the door hinge and limits; this **does not follow automatically** from the symbolic tuple.
4. The robot/environment state is updated by the connected motion/simulation system; **RViz displays** robot geometry, transforms and motion. RViz alone does not compute contact physics and does not guarantee that the real door opens.
5. Feedback/recording determines whether the observed state change occurred and whether constraints held during the **whole course**, including settling.

**Practical evidence still needed:** run the provided `setup_demo()` workflow in the linked Binder environment, show the relevant query bindings, execute the offered motion command using the notebook's actual API (rather than inventing a call), and attach a screenshot / recorded result. This submission does **not** claim that it has run RViz.

*Source: Exercise E11; Lecture 3 slides 14–15; Lecture 4 slides 5–16 and 38+; Lecture 4 student chapter section 1–2.*

### E12 · Ontological Limits & Missing Dynamics

A symbolic KB can associate an action (pull) with a change (open) and a capability claim for PR2. It cannot by itself determine whether a **wet carton will slip** under a certain acceleration: that needs mass, contact geometry, coefficient of friction, normal force, gravity and acceleration. For the simplified two-sided pinch, $2\mu F_n \ge m(g+a)$ is a **physical-law calculation**, not a DL classification. The semantic digital twin's **causal model of the controlled body–world system** predicts this transition; the planner/controller uses the predictions and measured feedback to select, realize and revise actions. Lecture 4 emphasizes that the **executive authorizes, the body realizes, and the coupled body–world system determines the outcome**.

*Source: Lecture 3 slide 34; Lecture 4 slides 2, 5–7, 22–23; Lecture 4 student chapter sections 1 and 3.*

# Submission guidelines

- Deadline: 24.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (If not yet, create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put the textual answers in notebook and all related media (images, rules, etc) should then be put into a folder called **Assignment3**. 

## Practical verification and submission checklist

- [x] All twelve textual answers revised against Lectures 1–4 and the four student chapters.
- [x] E1–E6 use the terminology of Lecture 3, including `ME ↔ [B ⋈ W]`, the external problem standard and priority-ordered classes.
- [x] E4 distinguishes direct `hasComponent` cardinality from transitive part relations.
- [x] E5 uses the actual lecture's RWMS value restriction and disjointness, not an invented contradiction.
- [x] E11 distinguishes visualization in RViz from physical simulation and trajectory execution.
- [ ] **Not yet completed:** run E7–E10 Prolog queries in the course Binder lab and paste actual bindings.
- [ ] **Not yet completed:** run `setup_demo()` and the notebook's actual motion interface, check PR2–fridge behavior, add a screenshot / execution record for E11.
- [ ] Confirm Git commit and push before **24 November 2026, 16:00**.

### Source documents used for the textual revision

- `slides1(2).pdf` and `student-chapter1(2).pdf` — Lecture 1, *Robots That Do the Task*.
- `slides2(1).pdf` and `student-chapter2(1).pdf` — Lecture 2, *The Body Motion Problem*.
- `slides3(1).pdf` and `student-chapter3(1).pdf` — Lecture 3, *The Robot Agent as a Scientific Object* (especially slides 10–11, 13–16, 18, 21–24, 26, 31–34, 37, 41).
- `slides4(1).pdf` and `student-chapter4(1).pdf` — Lecture 4, *The Physical Body–World System*.

**Scientific qualification:** the numerical examples in the lecture slides and the task sheet are illustrative unless specifically described as measurements. Symbolic derivability, planned motion and experimental evidence are distinct.